In [ ]:
# this notebook will normalize(scale)/ clean the data
# while keeping key level as feature
# revert file : Cmd+Shift+P 
import pandas as pd
import json
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "mlData"

symbol = "BTCUSDT"
tf = "5m"

folder_path = find_project_root() / "data" / "mlData" 
src_path = folder_path / "augmented" / f"{symbol}-{tf}-v8-augmented.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)

# extra col : experimenting data
# df["DELTA_2"] = df["DELTA_1"].rolling(2).sum()

print(df.columns)
df.head()

Index(['timestamp', 'label', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1',
       'DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV', 'ATR_5',
       'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_5', 'RSI_14',
       'RSI_SLOPE', 'STOCH_K', 'CCI_5', 'DIST_HIGH_5', 'DIST_LOW_5',
       'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS'],
      dtype='object')


,timestamp,label,ROC_3,ROC_5,ROC_10,MOM_3,RETURNS_1,DELTA_1,BUY_RATIO,DELTA_3,...,RSI_5,RSI_14,RSI_SLOPE,STOCH_K,CCI_5,DIST_HIGH_5,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS
0,1703910600000,NaN,0.000000,0.000000,0.000000,0.000000,NaN,5.491432,0.540597,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1703910900000,NaN,0.000371,0.000371,0.000371,0.421754,0.000371,4.745213,0.546678,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1703911200000,NaN,0.000418,0.000418,0.000418,0.475186,0.000047,-2.214020,0.474450,8.022625,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1703911500000,NaN,-0.000247,-0.000247,-0.000247,-0.281099,-0.000666,-9.078830,0.317680,-6.547637,...,64.801743,87.175247,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1703911800000,NaN,-0.001334,-0.000963,-0.000963,-1.515738,-0.000716,-10.343437,0.383228,-21.636287,...,43.995850,75.906944,NaN,0.024149,-140.554947,NaN,NaN,NaN,NaN,NaN


# Analyse Data
- Drop warm-up rows
- check Infinity, NaN, None
- verify data's quality. Non-staionary, local memory, etc.

In [ ]:
# removing warmup nan and check label imbalance
import numpy as np
# Replace inf with NaN 
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# list feature columns
group_I = ['ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1']
group_L = ['DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV']

feature_cols = [*group_I, *group_L]

# make sure that we drop only warm-up rows
df_clean = df.dropna(subset=feature_cols).reset_index(drop=True)

dropped = len(df) - len(df_clean)
print(f"Rows before : {len(df):,}")
print(f"Rows dropped: {dropped:,}  (warmup NaN from rolling windows)")
print(f"Rows after cleaned  : {len(df_clean):,}")


In [ ]:
# Trim start-up and tail NaN rows from label (only contiguous edges)
# also make sure there was not supposed to had any NaN left in any rows or columns

label = df_clean['label']

# Find first and last valid label index
first_valid_idx = label.first_valid_index()
last_valid_idx  = label.last_valid_index()

df_clean = df_clean.loc[first_valid_idx:last_valid_idx].reset_index(drop=True)

print(f"Label NaN trimmed — head up to index {first_valid_idx}, tail after index {last_valid_idx}")
print(f"Rows after trim: {len(df_clean):,}")

# Chronological order must still hold
assert df_clean['timestamp'].is_monotonic_increasing, "Timestamp order broken after trim!"

# Assert no NaN remains anywhere — only contiguous edge NaN in label were expected
remaining_nan = df_clean.isnull().sum()
remaining_nan = remaining_nan[remaining_nan > 0]
assert len(remaining_nan) == 0, f"Unexpected NaN still present:\n{remaining_nan.to_string()}"

print("Chronological order: OK")
print("All columns clean: no NaN remaining.")

# train all 3 labels
trainable = df_clean
n_up   = (trainable['label'] ==  1).sum()
n_down = (trainable['label'] == -1).sum()
total  = len(trainable)

print(f"\nLabel balance (trainable only):")
print(f"  Up   ( 1) : {n_up:,}  ({n_up   / total * 100:.1f}%)")
print(f"  Down (-1) : {n_down:,}  ({n_down / total * 100:.1f}%)")
print(f"  Timeout   : {(df_clean['label'] == 0).sum():,}")
print(f"  Total     : {total:,}  ({total / len(df_clean) * 100:.1f}% of clean rows)")


In [ ]:
# Nan and Infinity re-check
# import numpy as np

# 1. Feature columns must be clean (no NaN/Inf from warm-up)
assert df_clean[feature_cols].isnull().sum().sum() == 0, "NaN leaked through on feature columns!"
assert not np.isinf(df_clean[feature_cols].values).any(), "Inf leaked through on feature columns!"

# 2. Show any remaining NaN per column (expected only in 'label' from source data)
nan_counts = df_clean.isnull().sum()
nan_remaining = nan_counts[nan_counts > 0]
if len(nan_remaining) > 0:
    print("NaN remaining (source data NaN, NOT warm-up):")
    print(nan_remaining.to_string())
else:
    print("No NaN remaining in any column.")

# 3. Verify chronological order is preserved
assert df_clean['timestamp'].is_monotonic_increasing, "Timestamp order broken!"
print(f"\nChronological order: OK (timestamps monotonically increasing)")
print(f"Feature columns: clean (no NaN, no Inf)")


In [ ]:
print(df_clean.shape)
print(df_clean.columns)

# Test : Verify Data Quality
do them on df_clean
see [Data Quality](../../../../../journal/trainData/dataQuality.md)
- Are all features stationary
- Does it preserve local memory?
- Are features redundant with each other?
- Does it actually relate to the label?

# Skip to save Data if all had passed

In [ ]:
# Are all features stationary ? 
# ADF  rejects unit root     → not non-stationary
# KPSS rejects stationarity  → not trend-stationary
from statsmodels.tsa.stattools import adfuller, kpss
# import pandas as pd
# import numpy as np

def stationarity_report(series: pd.Series, name: str) -> dict:
    """
    ADF  null: series HAS a unit root (non-stationary)
         want: reject null → p < 0.05 → stationary

    KPSS null: series IS stationary
         want: fail to reject → p > 0.05 → stationary
    """
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(series.dropna(), autolag="AIC")
    kpss_stat, kpss_p, _, kpss_crit    = kpss(series.dropna(), regression="c", nlags="auto")

    verdict = "PASS" if (adf_p < 0.05 and kpss_p > 0.05) else \
              "WARN" if (adf_p < 0.05 or  kpss_p > 0.05) else "FAIL"

    return {
        "feature":   name,
        "adf_p":     round(adf_p,   4),
        "kpss_p":    round(kpss_p,  4),
        "adf_stat":  round(adf_stat, 3),
        "kpss_stat": round(kpss_stat,3),
        "verdict":   verdict,
    }

# Run on all Group I + L features
features_to_test = {
    "ROC_3":      df_clean["ROC_3"],
    "ROC_5":      df_clean["ROC_5"],
    "ROC_10":     df_clean["ROC_10"],
    "MOM_3":      df_clean["MOM_3"],
    "RETURNS_1":  df_clean["RETURNS_1"],
    "DELTA_1":    df_clean["DELTA_1"],
    # "DELTA_2":    df_clean["DELTA_2"],
    "DELTA_3":    df_clean["DELTA_3"],
    "BUY_RATIO":  df_clean["BUY_RATIO"],
    "VOL_SPIKE":  df_clean["VOL_SPIKE"],
    "DELTA_DIV":  df_clean["DELTA_DIV"],
}

# one-by-one test
# stationarity_report(df["RETURNS_1"],"RETURNS_1")

In [ ]:
# execute
# results = [stationarity_report(series, name) for name, series in features_to_test.items()]
# pd.DataFrame(results)

In [ ]:
# Delta div 1000 rolling window
# Investigate structural shift
# find out whether the divergence frequency changes across the sample
# plot rolling frequency of DELTA_DIV = 1 over time
window = 1000

rolling_freq = df["DELTA_DIV"].rolling(window).mean()

import matplotlib.pyplot as plt
plt.figure(figsize=(14, 4))
plt.plot(df.index, rolling_freq)
plt.axhline(y=rolling_freq.mean(), color="red",
            linestyle="--", label="global mean")
plt.title("DELTA_DIV Rolling Frequency (window=1000 bars)")
plt.ylabel("Fraction of bars with divergence")
plt.xlabel("Time")
plt.legend()
plt.tight_layout()
plt.savefig("delta_div_freq.png", dpi=150)
plt.show()

print(f"Global mean:  {df['DELTA_DIV'].mean():.4f}")
print(f"First half:   {df['DELTA_DIV'].iloc[:len(df)//2].mean():.4f}")
print(f"Second half:  {df['DELTA_DIV'].iloc[len(df)//2:].mean():.4f}")

# Two possible outcomes: DELTA_DIV
```
Outcome A — frequency is stable around global mean:
  → KPSS stat inflated by clustering, not a real shift
  → Keep DELTA_DIV as-is

Outcome B — frequency shifts significantly between halves:
  → Real structural change in market regime
  → Consider dropping or replacing with a rolling
     divergence rate feature instead of binary flag
```

In [ ]:
# revised pipeline
ROLLING_WINDOW = 500

features_rolling_z = ["DELTA_1", "DELTA_3", "BUY_RATIO"]

for feat in features_rolling_z:
    mean = df_clean[feat].rolling(ROLLING_WINDOW).mean()
    std  = df_clean[feat].rolling(ROLLING_WINDOW).std()
    df_clean[f"{feat}_rz"] = (df_clean[feat] - mean) / std.replace(0, np.nan)


# drop first 500 rows where rolling stats are unstable
tst = df_clean.iloc[ROLLING_WINDOW:].reset_index(drop=True)

rz_test = {
    "DELTA_1_rz_test":      tst["DELTA_1_rz"],
    "DELTA_3_rz_test":      tst["DELTA_3_rz"],
    "BUY_RATIO_rz_test":    tst["BUY_RATIO_rz"]
}

results = [stationarity_report(series, name) for name, series in rz_test.items()]
pd.DataFrame(results)

# Stationary Test

In [ ]:
# local memory test
# Sample Size Inflates Ljung-Box. Use another method

from statsmodels.tsa.stattools import acf
# import pandas as pd
# import numpy as np

def acf_magnitude_report(series: pd.Series, name: str, max_lag: int = 20):
    """
    Look at actual ACF coefficients, not just significance flags.
    Thresholds:
        |r| > 0.10  → meaningful memory
        |r| > 0.05  → weak but present
        |r| < 0.02  → negligible (likely noise even if p < 0.05)
    """
    acf_vals, confint = acf(series.dropna(), nlags=max_lag,
                            alpha=0.05, fft=True)

    # skip lag 0 (always 1.0)
    lags      = range(1, max_lag + 1)
    acf_lags  = acf_vals[1:]
    ci_lower  = confint[1:, 0] - acf_vals[1:]
    ci_upper  = confint[1:, 1] - acf_vals[1:]

    meaningful = [lag for lag, r in zip(lags, acf_lags) if abs(r) > 0.05]
    strong     = [lag for lag, r in zip(lags, acf_lags) if abs(r) > 0.10]

    print(f"\n{name}")
    print(f"  ACF at lag 1:  {acf_lags[0]:+.4f}")
    print(f"  ACF at lag 3:  {acf_lags[2]:+.4f}")
    print(f"  ACF at lag 5:  {acf_lags[4]:+.4f}")
    print(f"  ACF at lag 10: {acf_lags[9]:+.4f}")
    print(f"  ACF at lag 20: {acf_lags[19]:+.4f}")
    print(f"  Lags with |r| > 0.05 (weak):     {meaningful}")
    print(f"  Lags with |r| > 0.10 (meaningful): {strong}")

    return pd.DataFrame({
        "lag": list(lags),
        "acf": acf_lags.round(4),
    })

# run on all features
# for name, series in features_to_test.items():
#     acf_magnitude_report(series, name)

# extra check: till lag 50
# somehow DELTA_3 hasn't decay to zero
acf_magnitude_report(df["DELTA_3"], "DELTA_3", max_lag=50)

# Sample Report : Local memory test

In [ ]:
# Cross correlation test - Are features redundant with each other ?
import seaborn as sns

def correlation_report(df_features: pd.DataFrame, method: str = "spearman"):
    """
    Spearman preferred over Pearson — does not assume linearity.
    Flag pairs with |r| > 0.85 as redundant.
    """
    corr = df_features.corr(method=method)

    # find high-correlation pairs
    pairs = []
    cols  = corr.columns.tolist()
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            r = corr.iloc[i, j]
            if abs(r) > 0.85:
                pairs.append({"feature_a": cols[i],
                              "feature_b": cols[j],
                              "spearman_r": round(r, 3)})

    if pairs:
        print("High correlation pairs (|r| > 0.85) — consider dropping one:")
        print(pd.DataFrame(pairs).to_string(index=False))
    else:
        print("No redundant pairs found.")

    # heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
                center=0, vmin=-1, vmax=1)
    plt.title(f"Feature Correlation ({method.title()})")
    plt.tight_layout()
    plt.savefig("feature_correlation.png", dpi=150)
    plt.show()

    return corr

feature_df = pd.DataFrame({name: series for name, series in features_to_test.items()})
correlation_report(feature_df)

In [ ]:
# VIF : variance_inflation_factor
# Further Verification
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
# import numpy as np

def vif_report(X: np.ndarray, feature_names: list):
    vif = pd.DataFrame({
        "feature": feature_names,
        "VIF": [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    }).sort_values("VIF", ascending=False)
    print(vif.to_string(index=False))
    return vif

feature_names = [
    "ROC_3", "ROC_5", "ROC_10", "MOM_3", "RETURNS_1",   # Group I
    "DELTA_1", "DELTA_3", "BUY_RATIO", "VOL_SPIKE", "DELTA_DIV",  # Group L
]

# build the array from your dataframe
X_raw = df_clean[feature_names].dropna().values          # [n_samples, 10]

# Z-score first — VIF on raw unscaled features is misleading
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)             # [n_samples, 10]

# run it
vif_result = vif_report(X_scaled, feature_names)

# Sample Report : Feature Correlation test

In [ ]:
# MI : Mutual information 
# predictive signal
# MI values of 0.001–0.016 look alarming but are normal for financial time series.
# like NLP or image-classification
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing     import StandardScaler
import numpy as np

def mutual_info_report(X: np.ndarray, y: np.ndarray, feature_names: list):
    """
    Mutual information measures non-linear dependence between feature and label.
    Higher = more signal. Near 0 = feature is uninformative.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    mi = mutual_info_classif(X_scaled, y, random_state=42)

    report = pd.DataFrame({
        "feature": feature_names,
        "mutual_info": mi.round(4),
    }).sort_values("mutual_info", ascending=False)

    print(report.to_string(index=False))
    return report

feature_names = list(features_to_test.keys())
X = np.column_stack([s.values for s in features_to_test.values()])
y = df_clean["label"].values   # {0, 1, 2} encoded

mutual_info_report(X, y, feature_names)

# Note
DELTA_3 has extreme outliers — the range spans nearly 5000 units while 50% of values sit between -28 and +20. When mutual_info_classif builds its k-NN graph with default n_neighbors=3, the extreme outliers dominate the distance metric and the estimator fails to find local structure near the label boundary.

On the other hand, BUY_RATIO is tightly bounded. With n_neighbors=3, the 3 nearest neighbours of any point are so close together (tiny distances in a [0,1] space with std=0.15) that the estimator cannot distinguish label-relevant structure from noise at that resolution.

In [ ]:
# Diagnose why these two return zero

print(df_clean["DELTA_3"].describe())
print(df_clean["BUY_RATIO"].describe())

# Check for near-constant variance
print("\nDELTA_3  std:", df_clean["DELTA_3"].std())
print("BUY_RATIO std:", df_clean["BUY_RATIO"].std())

# Check distribution shape
print("\nDELTA_3  nunique:", df_clean["DELTA_3"].nunique())
print("BUY_RATIO nunique:", df_clean["BUY_RATIO"].nunique())

# Check NaN count
print("\nDELTA_3  NaN:", df_clean["DELTA_3"].isna().sum())
print("BUY_RATIO NaN:", df_clean["BUY_RATIO"].isna().sum())

In [ ]:
# Verify by tweaking k-NN
# and
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler, QuantileTransformer
# import numpy as np
# import pandas as pd

# feature_names = [
#     "ROC_3", "ROC_5", "ROC_10", "MOM_3", "RETURNS_1",
#     "DELTA_1", "DELTA_3", "BUY_RATIO", "VOL_SPIKE", "DELTA_DIV",
# ]

X_raw = df_clean[feature_names].dropna().values
y     = df_clean["label"].dropna().values

# ── Fix 1: increase n_neighbors ───────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

mi_k10 = mutual_info_classif(X_scaled, y, n_neighbors=10, random_state=42)

# ── Fix 2: quantile transform before MI ───────────────────────────────────────
# Maps each feature to uniform [0,1] distribution — removes outlier distortion
# and resolves the density problem in BUY_RATIO simultaneously
qt      = QuantileTransformer(output_distribution="uniform", random_state=42)
X_qt    = qt.fit_transform(X_raw)

mi_qt = mutual_info_classif(X_qt, y, n_neighbors=3, random_state=42)
mi_k3_orig = mutual_info_classif(X_scaled, y, n_neighbors=3, random_state=42)

report = pd.DataFrame({
    "feature":    feature_names,       # must match X column order exactly
    "mi_k3_orig": mi_k3_orig.round(4),
    "mi_k10":     mi_k10.round(4),
    "mi_qt_k3":   mi_qt.round(4),
}).sort_values("mi_qt_k3", ascending=False)

In [ ]:
# report

In [ ]:
# Final test
from scipy.stats import spearmanr

for name in ["DELTA_3", "BUY_RATIO"]:
    r, p = spearmanr(df_clean[name].dropna(), df_clean["label"].dropna())
    print(f"{name}  spearman_r={r:.4f}  p={p:.4f}")

# Export cleaned data

In [ ]:
df_clean.head()

In [ ]:
# save the current data in ./data/mlData/clean-v3
# save to JSONL for training
out_path = folder_path / "clean" / f"{symbol}-{tf}-v8-cleaned.jsonl"

df_clean.to_json(out_path, orient="records", lines=True)
print(f"final shape : {df_clean.shape}")
print(f"Saved {len(df_clean)} rows to {out_path}")
